In [1]:
!rm -rf TIGeR-Text-Image-Generative-Repair
!git clone https://github.com/namaray/TIGeR-Text-Image-Generative-Repair.git
%cd TIGeR-Text-Image-Generative-Repair
# Uninstall old version to clear Kaggle cache without breaking Torchvision
!pip uninstall -y tiger
!pip install --no-cache-dir -e ".[dev,vlm,gen]" -q
# Confirm import-abo is registered
!python -m tiger.cli --help | grep import

## 1. Import ABO Dataset
Make sure you have added the `khyeh0719/amazon-berkeley-objects-small` dataset to this notebook.

In [2]:
!python -m tiger.cli import-abo \
    --listings-dir /kaggle/input/datasets/khyeh0719/amazon-berkeley-objects-small/abo-listings/listings/metadata \
    --images-csv /kaggle/input/datasets/khyeh0719/amazon-berkeley-objects-small/abo-images-small/images/metadata/images.csv \
    --images-dir /kaggle/input/datasets/khyeh0719/amazon-berkeley-objects-small/abo-images-small/images/small


Importing ABO from:
  listings : /kaggle/input/datasets/khyeh0719/amazon-berkeley-objects-small/abo-listings/listings/metadata
  images   : /kaggle/input/datasets/khyeh0719/amazon-berkeley-objects-small/abo-images-small/images/metadata/images.csv
  img dir  : /kaggle/input/datasets/khyeh0719/amazon-berkeley-objects-small/abo-images-small/images/small
category     split      
electronics  calibration    2300
             report         2244
furniture    calibration     147
             report          193
home_decor   calibration      53
             report           63


### Validate Data Import
This ensures the notebook stops immediately if the ABO data wasn't found or parsed correctly.

In [3]:
import pandas as pd
from pathlib import Path

parquet_file = Path('data/sample/products.parquet')
assert parquet_file.exists(), "Data extraction failed: products.parquet not found! Check your --listings-dir and --images-dir paths."

df = pd.read_parquet(parquet_file)
print(f"Successfully imported {len(df)} products.")
assert len(df) > 0, "Data extraction failed: Parquet file is empty!"


Successfully imported 5000 products.


## 2. Calibrate on ABO
This fits the new similarity thresholds for the non-fashion domain.

In [4]:
!python -m tiger.cli calibrate

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2535, in __getattr__
    module = self._get_module(self._class_to_module[name])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2769, in _get_module
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2767, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importl

## 3. Inject Synthetic Noise
This injects errors into the report split so there is something to repair. **This step is required** — without it, the Arbiter and ablation have no corrupted products to work on.

In [5]:
!python -m tiger.cli noise --seed 7

wrote /kaggle/working/TIGeR-Text-Image-Generative-Repair/data/processed/noisy_report_seed7.parquet
{
  "seed": 7,
  "copies_per_row": 1,
  "rates": {
    "swap_image": 0.1,
    "swap_image_same_category": 0.03,
    "color_flip": 0.06,
    "near_color_flip": 0.02,
    "material_flip": 0.02,
    "attribute_drop": 0.02,
    "title_contradiction": 0.02,
    "mixed_swap_color": 0.02,
    "missing_image": 0.01
  },
  "rows_total": 2500,
  "rows_noisy": 750,
  "by_label": {
    "clean": 1750,
    "mutate_text": 350,
    "swap_image": 325,
    "mixed": 50,
    "missing_image": 25
  },
  "by_subtype": {
    "swap_image": 250,
    "color_flip": 150,
    "swap_image_same_category": 75,
    "attribute_drop": 50,
    "title_contradiction": 50,
    "near_color_flip": 50,
    "mixed_swap_color": 50,
    "material_flip": 50,
    "missing_image": 25
  },
  "self_verified": true
}


## 4. Retrain Arbiter
This retrains the Logistic Regression router on the new ABO-domain noise patterns.

In [6]:
!python -m tiger.cli train-arbiter

wrote /kaggle/working/TIGeR-Text-Image-Generative-Repair/data/processed/noisy_report_cal_seed1007.parquet
{
  "seed": 1007,
  "copies_per_row": 1,
  "rates": {
    "swap_image": 0.1,
    "swap_image_same_category": 0.03,
    "color_flip": 0.06,
    "near_color_flip": 0.02,
    "material_flip": 0.02,
    "attribute_drop": 0.02,
    "title_contradiction": 0.02,
    "mixed_swap_color": 0.02,
    "missing_image": 0.01
  },
  "rows_total": 2500,
  "rows_noisy": 750,
  "by_label": {
    "clean": 1750,
    "mutate_text": 350,
    "swap_image": 325,
    "mixed": 50,
    "missing_image": 25
  },
  "by_subtype": {
    "swap_image": 250,
    "color_flip": 150,
    "swap_image_same_category": 75,
    "material_flip": 50,
    "attribute_drop": 50,
    "near_color_flip": 50,
    "title_contradiction": 50,
    "mixed_swap_color": 50,
    "missing_image": 25
  },
  "self_verified": true
}
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils

## 5. Run Repair Ablation
This evaluates the repair pipeline using the Independent Verifier (SigLIP) and Generative Fallback.

In [7]:
!python -m tiger.cli ablate-repair --independent --generative-fallback

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2535, in __getattr__
    module = self._get_module(self._class_to_module[name])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2769, in _get_module
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2767, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importl

## 6. View Results
Compare this table with the Fashion results in your paper.

In [8]:
import pandas as pd
import glob
# Dynamically find whichever ablation CSV was produced
csvs = sorted(glob.glob('data/outputs/repair_ablations_summary*.csv'))
print('Found CSVs:', csvs)
if csvs:
    df = pd.read_csv(csvs[-1])
    print(df.to_string())
else:
    print('No results CSV found. Check that ablate-repair ran successfully.')

Found CSVs: []
No results CSV found. Check that ablate-repair ran successfully.
